In [1]:
!pip install datasets
!pip install transformers
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which 

In [2]:
from datasets import load_dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torch
from jiwer import wer


In [3]:
librispeech = load_dataset("RaphaelOlivier/librispeech_asr_adversarial", "adv", split='natural')

model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")


The repository for RaphaelOlivier/librispeech_asr_adversarial contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/RaphaelOlivier/librispeech_asr_adversarial.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating natural split: 0 examples [00:00, ? examples/s]

Generating adv_0.04 split: 0 examples [00:00, ? examples/s]

Generating adv_0.015 split: 0 examples [00:00, ? examples/s]

Generating adv_0.015_RIR split: 0 examples [00:00, ? examples/s]

...

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

In [4]:
import IPython.display as ipd
example = librispeech[10]

audio_array = example['audio']['array']

display(ipd.Audio(audio_array, rate=16000))
print(example["true_text"])



IT IS THIS THAT IS OF INTEREST TO THEORY OF KNOWLEDGE


In [7]:
import torch

def transcribe_audio(audio_array, sampling_rate, processor, model):
    if sampling_rate != 16000:
        raise ValueError(f"Expected 16kHz audio but got {sampling_rate}Hz")

    # Pass sampling_rate to the processor
    inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt", padding="longest")
    input_values = inputs.input_values

    with torch.no_grad():
        logits = model(input_values).logits
        predicted_ids = torch.argmax(logits, dim=-1)
        transcription = processor.batch_decode(predicted_ids)[0]

    return transcription


In [8]:
predicted_transcription = transcribe_audio(audio_array, 16000, processor, model)
print(predicted_transcription)

IT IS THIS THAT IS OF INTEREST TO THEORY OF KNOWLEDGE


## Model performence on clean data using WER
The **Word Error Rate (WER)** is a standard metric used to evaluate the performance of automatic speech recognition (ASR) systems. It measures the accuracy of a system's transcribed text by comparing it to the correct (ground truth) transcription. WER quantifies the number of errors in the transcription, expressed as a percentage or a normalized value, based on three types of errors:

- **Substitutions (S)**: Words in the transcription that are incorrectly replaced with different words.
- **Insertions (I)**: Extra words in the transcription that aren't in the ground truth.
- **Deletions (D)**: Words from the ground truth that are missing in the transcription.

### WER Formula

WER is calculated as:

\[
\text{WER} = \frac{S + I + D}{N}
\]

Where:

- \( S \): Number of substitutions.
- \( I \): Number of insertions.
- \( D \): Number of deletions.
- \( N \): Total number of words in the ground truth transcription.

The result is often multiplied by 100 to express WER as a percentage (e.g., WER = 0.25 means 25% error rate).

### Example

- **Ground Truth**: "the quick brown fox"
- **Transcription**: "the quick red fox jumps"
- **Alignment**:
  - "the" → "the" (correct)
  - "quick" → "quick" (correct)
  - "brown" → "red" (substitution: 1)
  - "fox" → "fox" (correct)
  - (none) → "jumps" (insertion: 1)
- **Calculation**:
  - \( S = 1 \), \( I = 1 \), \( D = 0 \)
  - \( N = 4 \) (words in ground truth)
  - \( \text{WER} = \frac{1 + 1 + 0}{4} = 0.5 \) (or 50%)



In [18]:

from jiwer import wer
from tqdm import tqdm
from statistics import mean



# Initialize list to store individual WERs
wers = []

# Process each sample with a progress bar
for example in tqdm(librispeech, desc="Processing LibriSpeech test set"):
    # Extract audio and ground truth
    audio_array = example["audio"]["array"]
    sampling_rate = example["audio"]["sampling_rate"]
    ground_truth = example["true_text"].lower().strip()

    # Generate transcription using the provided function
    try:
        transcription = transcribe_audio(audio_array, sampling_rate, processor, model).lower().strip()
    except Exception as e:
        print(f"Error transcribing sample: {e}")
        continue

    # Compute WER for this sample
    sample_wer = wer(ground_truth, transcription)
    wers.append(sample_wer)

# Calculate and display the average WER
if wers:
    average_wer = mean(wers)
    print()
    print(f"Average WER on LibriSpeech test set: {average_wer:.4f} ({average_wer * 100:.2f}%)")
else:
    print("No samples were successfully processed.")

# Optionally, save WERs for further analysis
import json
wer_results = {"individual_wers": wers, "average_wer": average_wer if wers else None}
with open("librispeech_test_wer.json", "w") as f:
    json.dump(wer_results, f, indent=2)

Processing LibriSpeech test set: 100%|██████████| 85/85 [01:10<00:00,  1.20it/s]


Average WER on LibriSpeech test set: 0.0368 (3.68%)


## Applying FGSM attack And Getting Average WER

In [19]:
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from datasets import load_dataset
import jiwer
import numpy as np

def fgsm_attack(audio_array, ground_truth, target_transcription, model, processor, epsilon=0.3, sampling_rate=16000):
    """
    Perform FGSM attack on Wav2Vec2 model to generate adversarial audio.

    Args:
        audio_array (np.ndarray): Input audio waveform (1D NumPy array).
        ground_truth (str): Ground truth transcription.
        target_transcription (str): Desired target transcription for the attack.
        model (Wav2Vec2ForCTC): Pre-trained Wav2Vec2 model.
        processor (Wav2Vec2Processor): Wav2Vec2 processor for audio and text processing.
        epsilon (float): Perturbation magnitude for FGSM. Default is 0.3.
        sampling_rate (int): Audio sampling rate (default: 16000 Hz).
        device (str): Device to run the model on (default: "cuda").

    Returns:
        tuple: (adversarial_waveform, ground_truth_wer, target_wer, adversarial_transcription)
            - adversarial_waveform (np.ndarray): Perturbed audio waveform.
            - ground_truth_wer (float): WER between ground truth and adversarial transcription.
            - target_wer (float): WER between target transcription and adversarial transcription.
            - adversarial_transcription (str): Transcription of the adversarial audio.
    """
    # Step 1: Preprocess the audio
    inputs = processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt", padding="longest")
    input_values = inputs.input_values  # Shape: [1, audio_length]

    # Step 2: Tokenize the target transcription
    labels = processor.tokenizer(target_transcription, return_tensors="pt").input_ids

    # Step 3 & 4: Compute CTC loss with gradient tracking
    input_values.requires_grad_(True)
    output = model(input_values, labels=labels)
    loss = output.loss
    loss.backward()

    # Step 5: Compute the gradient
    grad = input_values.grad  # Gradient of loss w.r.t. input_values

    # Step 6: Generate perturbation (targeted attack: minimize loss w.r.t. target)
    perturbation = -epsilon * torch.sign(grad)

    # Step 7: Create adversarial input
    adversarial_input_values = input_values.detach() + perturbation

    # Step 8: Transcribe adversarial audio
    with torch.no_grad():
        logits = model(adversarial_input_values).logits
        predicted_ids = torch.argmax(logits, dim=-1)
        adversarial_transcription = processor.batch_decode(predicted_ids)[0]

    # Step 9: Evaluate the attack
    ground_truth_wer = jiwer.wer(ground_truth, adversarial_transcription)
    target_wer = jiwer.wer(target_transcription, adversarial_transcription)

    # Convert adversarial input to NumPy array for return
    adversarial_waveform = adversarial_input_values.squeeze().cpu().numpy()

    return adversarial_waveform, ground_truth_wer, target_wer, adversarial_transcription

In [20]:
example = librispeech[11]
audio_array = example["audio"]["array"]  # Raw audio waveform
ground_truth = example["true_text"]  # Ground truth transcription
target_transcription = "HELLO WORLD"  # Target transcription

# Run FGSM attack
adversarial_waveform, ground_truth_wer, target_wer, adversarial_transcription = fgsm_attack(
    audio_array=audio_array,
    ground_truth=ground_truth,
    target_transcription=target_transcription,
    model=model,
    processor=processor,
    epsilon=0.02
)

# Print results
print(f"Ground Truth             : {ground_truth}")
print(f"Adversarial Transcription: {adversarial_transcription}")
print(f"WER (Ground Truth): {ground_truth_wer:.2f}")

Ground Truth             : AS USED IN THE SPEECH OF EVERYDAY LIFE THE WORD CARRIES AN UNDERTONE OF DEPRECATION
Adversarial Transcription: AS USE IN THE SPEECH OF EVERYDAY LIFE THE WORD CARIES AN UNDERTONE OF DEPRECATION
WER (Ground Truth): 0.13


In [21]:
import IPython.display as ipd
print("Original Audio:")
display(ipd.Audio(audio_array, rate=16000))
adversarial_audio = adversarial_waveform
print("Adversarial Audio:")
display(ipd.Audio(adversarial_audio, rate=16000))

Original Audio:


Adversarial Audio:


In [ ]:
from statistics import mean
import json



# Define epsilon values to test (from 0.001 to 0.3)
epsilon_values = [0.01, 0.02, 0.05, 0.1, 0.2]

# Define target transcription
target_transcription = "HELLO WORLD"

# Store results
results = []
wer_by_epsilon = {eps: {"ground_truth_wer": [], "target_wer": []} for eps in epsilon_values}

# Select three samples for audio playback (e.g., indices 0, 1, 2)
audio_samples_to_play = [0, 1, 2]  # Adjust if you want different indices
audio_results = {idx: {eps: {} for eps in epsilon_values} for idx in audio_samples_to_play}

# Loop over the dataset
for idx, example in enumerate(librispeech):
    audio_array = example["audio"]["array"]  # Raw audio waveform
    ground_truth = example["true_text"]  # Ground truth transcription

    if idx % 10 == 0:
      print(f"\nSample {idx}")

    # Loop over epsilon values
    for epsilon in epsilon_values:
        # Run FGSM attack
        adversarial_waveform, ground_truth_wer, target_wer, adversarial_transcription = fgsm_attack(
            audio_array=audio_array,
            ground_truth=ground_truth,
            target_transcription=target_transcription,
            model=model,
            processor=processor,
            epsilon=epsilon
        )

        # Store result (exclude waveform to avoid JSON serialization issue)
        result = {
            "sample_idx": idx,
            "epsilon": epsilon,
            "ground_truth": ground_truth,
            "adversarial_transcription": adversarial_transcription,
            "ground_truth_wer": ground_truth_wer,
            "target_wer": target_wer
        }
        results.append(result)

        # Collect WERs for averaging
        wer_by_epsilon[epsilon]["ground_truth_wer"].append(ground_truth_wer)
        wer_by_epsilon[epsilon]["target_wer"].append(target_wer)

        # Store audio results for playback if sample is selected
        if idx in audio_samples_to_play:
            audio_results[idx][epsilon] = {
                "adversarial_waveform": adversarial_waveform,
                "ground_truth": ground_truth,
                "adversarial_transcription": adversarial_transcription,
                "ground_truth_wer": ground_truth_wer,
            }


In [24]:
# Play adversarial waveforms for selected samples
print("\nPlaying Adversarial Waveforms for Selected Samples:")
for idx in audio_samples_to_play:
    print(f"\nSample {idx}:")
    for epsilon in epsilon_values:
        audio_data = audio_results[idx][epsilon]
        print(f"\nEpsilon: {epsilon}")
        print(f"Ground Truth: {audio_data['ground_truth']}")
        print(f"Adversarial Transcription: {audio_data['adversarial_transcription']}")
        print(f"WER (Ground Truth): {audio_data['ground_truth_wer']:.2f}")
        print("Playing Adversarial Audio:")
        display(ipd.Audio(audio_data['adversarial_waveform'], rate=16000))

# Compute and print average WER for each epsilon
print("\nAverage WER Across All Samples:")
for epsilon in epsilon_values:
    avg_ground_truth_wer = mean(wer_by_epsilon[epsilon]["ground_truth_wer"])
    avg_target_wer = mean(wer_by_epsilon[epsilon]["target_wer"])
    print(f"Epsilon: {epsilon}")
    print(f"  Average Ground Truth WER: {avg_ground_truth_wer:.2f}")
    print(f"  Average Target WER: {avg_target_wer:.2f}")

# Save results to JSON
with open("fgsm_results.json", "w") as f:
    json.dump(results, f, indent=2)


Playing Adversarial Waveforms for Selected Samples:

Sample 0:

Epsilon: 0.01
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
WER (Ground Truth): 0.00
Playing Adversarial Audio:



Epsilon: 0.02
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
WER (Ground Truth): 0.00
Playing Adversarial Audio:



Epsilon: 0.05
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
WER (Ground Truth): 0.00
Playing Adversarial Audio:



Epsilon: 0.1
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
WER (Ground Truth): 0.00
Playing Adversarial Audio:



Epsilon: 0.2
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
WER (Ground Truth): 0.00
Playing Adversarial Audio:


...


Epsilon: 0.02
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: BUT I DO NOT THINK SUCH AN INFERENCE IS WARN'T IT
WER (Ground Truth): 0.20
Playing Adversarial Audio:



Epsilon: 0.05
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: BUT I DO NOT THINK SUCH AN IPERENCE IS WARRANTED
WER (Ground Truth): 0.10
Playing Adversarial Audio:



Epsilon: 0.1
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: BUT I DO NOT THINK SUCH AN IPERENCE IS WARRANTED
WER (Ground Truth): 0.10
Playing Adversarial Audio:



Epsilon: 0.2
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: BUT I DO NOT THINK SUCH AN EQENT IS WANTED
WER (Ground Truth): 0.20
Playing Adversarial Audio:



Average WER Across All Samples:
Epsilon: 0.01
  Average Ground Truth WER: 0.09
  Average Target WER: 6.48
Epsilon: 0.02
  Average Ground Truth WER: 0.09
  Average Target WER: 6.48
Epsilon: 0.05
  Average Ground Truth WER: 0.10
  Average Target WER: 6.48
Epsilon: 0.1
  Average Ground Truth WER: 0.11
  Average Target WER: 6.49
Epsilon: 0.2
  Average Ground Truth WER: 0.17
  Average Target WER: 6.50
